# Resume Maker AI Agent

made this using crewAI for my assignment.

basically made a small team of 3 agents:
1. analyzer - reads my resume and the job description and figures out gaps
2. writer - rewrites my resume to match the job
3. reviewer - checks the final resume and polishes it

run all the cells below and it will print the final resume + save it in final_resume.txt

note: jupyter runs its own event loop in the background, so we have to use
`kickoff_async()` with `await` instead of the normal `kickoff()`. jupyter
supports `await` directly in a cell, no extra setup needed for that.

### 1. imports and setup

In [16]:
import os
from crewai import Agent, Task, Crew, Process
from utils import get_groq_api_key

# ---- fix for a known crewai bug ----
# crewai tries to add a "cache_breakpoint" tag to messages (a feature made for
# openai/anthropic), but groq doesn't support that tag and rejects the request.
# this turns that tagging off so groq stops complaining.
# more info: https://github.com/crewAIInc/crewAI/issues/5886
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

# load api key from .env file (utils.py handles the dotenv loading)
os.environ["GROQ_API_KEY"] = get_groq_api_key()

# using groq's free llama model for all agents
my_llm = "groq/llama-3.3-70b-versatile"

### 2. load my resume and the job description

(put your own resume and job desc in these files before running, sample ones are already there)

In [17]:
with open("resume.txt", "r") as f:
    my_resume = f.read()

with open("job_description.txt", "r") as f:
    job_desc = f.read()

print(my_resume)  # just checking it loaded fine

Shaurya
Email: shauryagoyal89@gmail.com | Phone: 9116616700

Education:
B.Tech, Chemical Science and Technology (ongoing)

Skills:
Python, Machine Learning basics, SQL, R, C/C++

Projects:
- Farmer Income Prediction Model - used XGBoost, Random Forest and Ridge regression, combined using ensemble blending
- Grid Electricity Demand Forecasting (Predictive Paradox) - built using Python and scikit-learn for forecasting demand

Experience:
- Participated in IIT Guwahati Consulting and Analytics Club hackathon (ML task on NHANES dataset)



### 3. agents

In [18]:
analyzer = Agent(
    role="Resume Analyzer",
    goal="Compare the resume with the job description and find what is missing or weak",
    backstory=(
        "You used to work in HR for 5 years shortlisting resumes, "
        "so you know exactly what recruiters look for and what usually gets a resume rejected."
    ),
    llm=my_llm,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Resume Writer",
    goal="Rewrite the resume so it matches the job description better and highlights the right skills",
    backstory=(
        "You are a professional resume writer who has helped hundreds of people "
        "get shortlisted by making their resumes sound sharp and relevant."
    ),
    llm=my_llm,
    verbose=True,
    allow_delegation=False,
)

reviewer = Agent(
    role="Resume Reviewer",
    goal="Check the final resume for mistakes, weird phrasing, and make sure formatting is clean",
    backstory=(
        "You are a strict editor who hates grammar mistakes, filler words, "
        "and resumes that sound generic. You make sure everything reads well."
    ),
    llm=my_llm,
    verbose=True,
    allow_delegation=False,
)

### 4. tasks

In [19]:
analyze_task = Task(
    description=(
        f"Here is the candidate's current resume:\n{my_resume}\n\n"
        f"Here is the job description:\n{job_desc}\n\n"
        "Compare both and list out:\n"
        "- skills/keywords from the job description that are missing in the resume\n"
        "- weak points in the resume\n"
        "- what should be emphasized more"
    ),
    expected_output="A short list of gaps and suggestions to improve the resume for this job",
    agent=analyzer,
)

write_task = Task(
    description=(
        "Using the gaps and suggestions from the analyzer, rewrite the resume "
        "so it is tailored to the job description. Keep it realistic, don't add fake experience, "
        "just rephrase and reorganize to highlight relevant stuff. Keep the same sections "
        "(like Education, Experience, Skills, Projects) but improve the wording."
    ),
    expected_output="A full rewritten resume tailored to the job",
    agent=writer,
    context=[analyze_task],
)

review_task = Task(
    description=(
        "Review the rewritten resume. Fix grammar mistakes, remove repeated words, "
        "make sure formatting looks clean and consistent, and give the final polished version."
    ),
    expected_output="The final clean, polished resume, ready to send",
    agent=reviewer,
    context=[write_task],
)

### 5. crew - putting all agents and tasks together

In [20]:
resume_crew = Crew(
    agents=[analyzer, writer, reviewer],
    tasks=[analyze_task, write_task, review_task],
    process=Process.sequential,  # doing it step by step, one after another
    verbose=True,
)

### 6. run it

In [21]:
# using kickoff_async here because jupyter runs its own event loop,
# and crewai's normal kickoff() doesn't work inside that. this is jupyter-specific,
# a plain .py script can just use resume_crew.kickoff() instead
result = await resume_crew.kickoff_async()

print("\n\n--------- FINAL RESUME ---------\n")
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.1                                                                                        │
│  Latest version:  1.15.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 912787e0-9a86-4f48-8a0d-b2d152d2f915                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Here is the candidate's current resume:                                                                  │
│  Shaurya                                                                                                        │
│  Email: shauryagoyal89@gmail.com | Phone: 9116616700                                                            │
│                                                                                                                 │
│  Education:                                                                                                     │
│  B.Tech, Chemical Science and Technology (ongoing)                                                              │
│                                                                                                                 │
│  Skills:                                                                                                        │
│  Python, Machine Learning basics, SQL, R, C/C++                                                                 │
│                                                                                                                 │
│  Projects:                                                                                                      │
│  - Farmer Income Prediction Model - used XGBoost, Random Forest and Ridge regression, combined using ensemble   │
│  blending                                                                                                       │
│  - Grid Electricity Demand Forecasting (Predictive Paradox) - built using Python and scikit-learn for           │
│  forecasting demand                                                                                             │
│                                                                                                                 │
│  Experience:                                                                                                    │
│  - Participated in IIT Guwahati Consulting and Analytics Club hackathon (ML task on NHANES dataset)             │
│                                                                                                                 │
│                                                                                                                 │
│  Here is the job description:                                                                                   │
│  We are looking for a Data Analyst Intern with the following skills:                                            │
│  - Strong knowledge of Python and SQL                                                                           │
│  - Experience with machine learning models (regression, classification)                                         │
│  - Ability to work with large datasets and clean messy data                                                     │
│  - Good communication and presentation skills                                                                   │
│  - Familiarity with data visualization tools like Power BI or Tableau is a plus                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Compare both and list out:                                                                                     │
│  - skills/keywords from the job description that are missing in the resume                                      │
│  - weak points in the resume                           

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyzer                                                                                         │
│                                                                                                                 │
│  Task: Here is the candidate's current resume:                                                                  │
│  Shaurya                                                                                                        │
│  Email: shauryagoyal89@gmail.com | Phone: 9116616700                                                            │
│                                                                                                                 │
│  Education:                                                                                                     │
│  B.Tech, Chemical Science and Technology (ongoing)                                                              │
│                                                                                                                 │
│  Skills:                                                                                                        │
│  Python, Machine Learning basics, SQL, R, C/C++                                                                 │
│                                                                                                                 │
│  Projects:                                                                                                      │
│  - Farmer Income Prediction Model - used XGBoost, Random Forest and Ridge regression, combined using ensemble   │
│  blending                                                                                                       │
│  - Grid Electricity Demand Forecasting (Predictive Paradox) - built using Python and scikit-learn for           │
│  forecasting demand                                                                                             │
│                                                                                                                 │
│  Experience:                                                                                                    │
│  - Participated in IIT Guwahati Consulting and Analytics Club hackathon (ML task on NHANES dataset)             │
│                                                                                                                 │
│                                                                                                                 │
│  Here is the job description:                                                                                   │
│  We are looking for a Data Analyst Intern with the following skills:                                            │
│  - Strong knowledge of Python and SQL                                                                           │
│  - Experience with machine learning models (regression, classification)                                         │
│  - Ability to work with large datasets and clean messy data                                                     │
│  - Good communication and presentation skills                                                                   │
│  - Familiarity with data visualization tools like Power BI or Tableau is a plus                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Compare both and list out:                                                                                     │
│  - skills/keywords from the job description that are mi

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Reviewer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Your Name**                                                                                                  │
│  **Address**                                                                                                    │
│  **Phone Number**                                                                                               │
│  **Email**                                                                                                      │
│  **LinkedIn Profile (optional)**                                                                                │
│                                                                                                                 │
│  **Objective:**                                                                                                 │
│  To secure a Data Analyst Intern position where I can apply my skills in data analysis, machine learning, and   │
│  data visualization to drive business growth and improve decision-making.                                       │
│                                                                                                                 │
│  **Education:**                                                                                                 │
│  - Currently pursuing B.Tech in Computer Science, **University Name** (Expected Graduation: **Date**)           │
│    - Relevant coursework: Data Structures, Algorithms, Database Management Systems, Machine Learning, and       │
│  Statistics                                                                                                     │
│                                                                                                                 │
│  **Experience:**                                                                                                │
│  - **Data Analysis Project**, **Project Duration**                                                              │
│    - Collaborated with a team to analyze a dataset of **dataset size**, identifying trends and patterns, and    │
│  presented findings using **data visualization tools**                                                          │
│    - Utilized Python and SQL to clean and preprocess the data, applying machine learning models to predict      │
│  **target variable**                                                                                            │
│    - Gained valuable insights into the importance of data quality and the impact of data visualization on       │
│  storytelling                                                                                                   │
│  - **Hackathon Participation**, **Hackathon Name**, **Date**                                                    │
│    - Collaborated with a team to develop a predictive model using machine learning algorithms to solve a        │
│  real-world problem                                                                                             │
│    - Presented the project to a panel of judges, receiving feedback on the approach and implementation          │
│                                                                                                                 │
│  **Skills:**                                                                                                    │
│  - **Technical Skills:**                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the rewritten resume. Fix grammar mistakes, remove repeated words, make sure formatting looks     │
│  clean and consistent, and give the final polished version.                                                     │
│  Agent: Resume Reviewer                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 912787e0-9a86-4f48-8a0d-b2d152d2f915                                                                       │
│  Final Output: **Your Name**                                                                                    │
│  **Address**                                                                                                    │
│  **Phone Number**                                                                                               │
│  **Email**                                                                                                      │
│  **LinkedIn Profile (optional)**                                                                                │
│                                                                                                                 │
│  **Objective:**                                                                                                 │
│  To secure a Data Analyst Intern position where I can apply my skills in data analysis, machine learning, and   │
│  data visualization to drive business growth and improve decision-making.                                       │
│                                                                                                                 │
│  **Education:**                                                                                                 │
│  - Currently pursuing B.Tech in Computer Science, **University Name** (Expected Graduation: **Date**)           │
│    - Relevant coursework: Data Structures, Algorithms, Database Management Systems, Machine Learning, and       │
│  Statistics                                                                                                     │
│                                                                                                                 │
│  **Experience:**                                                                                                │
│  - **Data Analysis Project**, **Project Duration**                                                              │
│    - Collaborated with a team to analyze a dataset of **dataset size**, identifying trends and patterns, and    │
│  presented findings using **data visualization tools**                                                          │
│    - Utilized Python and SQL to clean and preprocess the data, applying machine learning models to predict      │
│  **target variable**                                                                                            │
│    - Gained valuable insights into the importance of data quality and the impact of data visualization on       │
│  storytelling                                                                                                   │
│  - **Hackathon Participation**, **Hackathon Name**, **Date**                                                    │
│    - Collaborated with a team to develop a predictive model using machine learning algorithms to solve a        │
│  real-world problem                                                                                             │
│    - Presented the project to a panel of judges, receiving feedback on the approach and implementation          │
│                                                                                                                 │
│  **Skills:**                                                                                                    │
│  - **Technical Skills:**                              



--------- FINAL RESUME ---------

**Your Name**
**Address**
**Phone Number**
**Email**
**LinkedIn Profile (optional)**

**Objective:**
To secure a Data Analyst Intern position where I can apply my skills in data analysis, machine learning, and data visualization to drive business growth and improve decision-making.

**Education:**
- Currently pursuing B.Tech in Computer Science, **University Name** (Expected Graduation: **Date**)
  - Relevant coursework: Data Structures, Algorithms, Database Management Systems, Machine Learning, and Statistics

**Experience:**
- **Data Analysis Project**, **Project Duration**
  - Collaborated with a team to analyze a dataset of **dataset size**, identifying trends and patterns, and presented findings using **data visualization tools**
  - Utilized Python and SQL to clean and preprocess the data, applying machine learning models to predict **target variable**
  - Gained valuable insights into the importance of data quality and the impact of data visua

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 7. save the final resume to a file so i dont lose it

In [22]:
with open("final_resume.txt", "w") as f:
    f.write(str(result))

print("saved to final_resume.txt")

# TODO: maybe later add a 4th agent that writes a cover letter too

saved to final_resume.txt
